In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import os
import warnings
warnings.filterwarnings("ignore")


df = pd.read_csv("FEATURES_PREPARED.csv")
df["Date"] = pd.to_datetime(df["Date"], utc=True)
df = df.sort_values("Date").reset_index(drop=True)

PM_FEATURES          = ["FED_DELTA", "GDP_DELTA", "UNEMPLOYMENT_DELTA", "INF_MONTHLY_DELTA"]
ANNOUNCEMENT_DUMMIES = ["Ann_CPI", "Ann_FOMC", "Ann_Employment", "Ann_GDP"]

CORE_ASSETS = {"BTC": "BTC_chg", "Gold": "Gold_chg", "Oil": "Oil_chg", "SPY": "SPY_chg"}
EXTENDED    = {"QQQ": "QQQ_chg", "SP500_fut": "SP500_fut_chg"}
ASSETS      = {**CORE_ASSETS, **EXTENDED}
ASSETS      = {k: v for k, v in ASSETS.items() if v in df.columns}

TC = {"SPY": 0.0002, "QQQ": 0.0002, "SP500_fut": 0.0001,
      "BTC": 0.0001, "Gold": 0.0001, "Oil": 0.0002}

ANNUAL_HOURS   = 252 * 24
ROLLING_WINDOW = 20

print(f"Assets: {list(ASSETS.keys())}")
print(f"Dataset: {df.shape[0]} rows, "
      f"{df['Date'].iloc[0].date()} → {df['Date'].iloc[-1].date()}")


HOLDING_PERIODS_DEFAULT = {
    ("FED_DELTA",          "SPY"):       2,
    ("FED_DELTA",          "QQQ"):       2,
    ("FED_DELTA",          "SP500_fut"): 2,
    ("FED_DELTA",          "BTC"):       2,
    ("FED_DELTA",          "Gold"):      2,
    ("INF_MONTHLY_DELTA",  "Oil"):       1,
    ("UNEMPLOYMENT_DELTA", "SPY"):       1,
    ("UNEMPLOYMENT_DELTA", "QQQ"):       1,
    ("UNEMPLOYMENT_DELTA", "SP500_fut"): 1,
}

IRF_FILE = "results/results_irf.csv"

if os.path.exists(IRF_FILE):
    irf_df = pd.read_csv(IRF_FILE)
    HOLDING_PERIODS = {}
    for _, row in irf_df.iterrows():
        asset_clean = row["asset"].replace("_chg", "")

        for k in ASSETS:
            if ASSETS[k] == row["asset"] or k == asset_clean:
                HOLDING_PERIODS[(row["pm_signal"], k)] = int(row["optimal_h"])

    for key, h in HOLDING_PERIODS_DEFAULT.items():
        if key[1] in EXTENDED and key not in HOLDING_PERIODS:
            HOLDING_PERIODS[key] = h
    print(f"\nHolding periods loaded from {IRF_FILE}:")
else:
    HOLDING_PERIODS = HOLDING_PERIODS_DEFAULT
    print(f"\nWARNING: {IRF_FILE} not found — using DLM-derived defaults.")

HOLDING_PERIODS = {k: v for k, v in HOLDING_PERIODS.items() if k[1] in ASSETS}
for (pm, asset), h in sorted(HOLDING_PERIODS.items()):
    src = "IRF" if os.path.exists(IRF_FILE) else "default"
    print(f"  {pm:30s} → {asset:12s}  h={h} [{src}]")



print("\n" + "="*65)
print("="*65)

for pm in PM_FEATURES:
    missing_col = f"{pm}_is_missing"


    mu  = df[pm].shift(1).rolling(ROLLING_WINDOW, min_periods=5).mean()
    sig = df[pm].shift(1).rolling(ROLLING_WINDOW, min_periods=5).std()


    shock_flag = df[pm].abs() > (mu + 2 * sig)

    df[f"{pm}_shock"] = shock_flag.astype(int)
    df[f"{pm}_z"]     = (df[pm] - mu) / (sig + 1e-10)


    if missing_col in df.columns:
        df.loc[df[missing_col] == 1, f"{pm}_shock"] = 0

    n = df[f"{pm}_shock"].sum()
    print(f"  {pm:30s}: {n} shocks ({100*n/len(df):.1f}%)")

ann_flag  = df[ANNOUNCEMENT_DUMMIES].any(axis=1)
any_shock = df[[f"{pm}_shock" for pm in PM_FEATURES]].any(axis=1)
overlap   = (ann_flag & any_shock).sum()
print(f"\n  Shocks overlapping announcements: {overlap} "
      f"({100*overlap/max(any_shock.sum(),1):.1f}% of all shocks)")
print("  → Confirms most shocks are genuinely intraday")


# EVENT STUDY

print("\n\n" + "="*65)
print("  SECTION 3C — EVENT STUDY")
print("="*65)

EVENT_PM_MAP = {
    "Ann_FOMC":       "FED_DELTA",
    "Ann_CPI":        "INF_MONTHLY_DELTA",
    "Ann_Employment": "UNEMPLOYMENT_DELTA",
    "Ann_GDP":        "GDP_DELTA",
}

PRE_HOURS  = 6
POST_HOURS = 3


# Activity test: PRE-announcement hours only

print("\nPre-announcement activity test (6h BEFORE announcement vs normal hours):")
print(f"  {'PM Signal':25s}  {'Event':20s}  {'n_events':>8}  {'Ratio':>8}  {'p-value':>10}")
print(f"  {'-'*78}")

activity_results = []
for ann_col, pm_col in EVENT_PM_MAP.items():
    ann_idx = df.index[df[ann_col] == 1].tolist()
    if len(ann_idx) < 3:
        continue

    # Strictly PRE: 6 hours before each announcement, not including ann hour
    pre_idx = set()
    for i in ann_idx:
        for j in range(max(0, i - PRE_HOURS), i):
            pre_idx.add(j)

    non_pre_idx = set(df.index) - pre_idx - set(ann_idx)

    event_vals  = df.loc[list(pre_idx),     pm_col].abs().dropna()
    normal_vals = df.loc[list(non_pre_idx), pm_col].abs().dropna()

    if len(event_vals) < 5 or len(normal_vals) < 5:
        continue

    ratio        = event_vals.mean() / (normal_vals.mean() + 1e-10)
    t_stat, pval = stats.ttest_ind(event_vals, normal_vals, equal_var=False)
    stars = "***" if pval<0.01 else "**" if pval<0.05 else "*" if pval<0.10 else ""
    print(f"  {pm_col:25s}  {ann_col:20s}  {len(ann_idx):8d}  "
          f"{ratio:8.2f}x  {pval:10.4f} {stars}")
    activity_results.append({
        "pm": pm_col, "event": ann_col, "n_events": len(ann_idx),
        "activity_ratio": round(ratio, 3), "pval_activity": round(pval, 4)
    })


# Predictive regression S_pre → R_post (HC3 robust SEs)

print(f"\n\nPredictive regressions (S_pre → R_post, HC3 robust SEs):")
print(f"  {'Event':20s}  {'PM':25s}  {'Asset':10s}  {'n':>4}  {'β':>10}  {'p(HC3)':>10}")
print(f"  {'-'*88}")

event_study_results = []

for ann_col, pm_col in EVENT_PM_MAP.items():
    ann_hours = df.index[df[ann_col] == 1].tolist()
    if len(ann_hours) < 5:
        continue


    asset_order = list(CORE_ASSETS.keys()) + [k for k in EXTENDED if k in ASSETS]
    for asset_name in asset_order:
        if asset_name not in ASSETS:
            continue
        asset_col = ASSETS[asset_name]

        pre_signals, post_returns = [], []
        for idx in ann_hours:
            pre_window  = df.loc[max(0, idx - PRE_HOURS) : idx - 1, pm_col]
            s_pre       = pre_window.sum()
            post_end    = min(len(df) - 1, idx + POST_HOURS)
            post_window = df.loc[idx + 1 : post_end, asset_col]
            r_post      = post_window.sum()
            if pd.isna(s_pre) or pd.isna(r_post) or post_window.isna().all():
                continue
            pre_signals.append(s_pre)
            post_returns.append(r_post)

        if len(pre_signals) < 5:
            continue

        try:
            X     = sm.add_constant(pre_signals)
            model = sm.OLS(post_returns, X).fit(cov_type="HC3")
            beta  = model.params[1]
            pval  = model.pvalues[1]
            n     = len(pre_signals)
            stars = ("***" if pval<0.01 else "**" if pval<0.05
                     else "*" if pval<0.10 else "~" if pval<0.15 else "")
            tag   = " [ext]" if asset_name in EXTENDED else ""
            print(f"  {ann_col:20s}  {pm_col:25s}  {asset_name+tag:10s}  "
                  f"{n:4d}  {beta:+10.4f}  {pval:10.4f} {stars}")
            event_study_results.append({
                "event": ann_col, "pm": pm_col, "asset": asset_name,
                "n_events": n, "beta": round(beta, 6), "pval_hc3": round(pval, 4)
            })
        except Exception:
            continue

if event_study_results:
    pd.DataFrame(event_study_results).to_csv("event_study_results.csv", index=False)
    print("\nSaved: event_study_results.csv")


# BACKTEST & SHARPE RATIO

print("\n\n" + "="*65)
print("  SECTION 5 — BACKTEST & SHARPE RATIO")
print("="*65)
print("""
Strategy:
  Entry: only at PM shock hours (aligned with Section 3B shocks)
  Direction: sign(PM shock at time t) — honest real-time rule
  Hold: h hours cumulative return (t+1 through t+h) from IRF
  No overlapping trades: block new entries until current trade closes
  Skip: any trade where the full h-hour window has NaN (e.g. SPY overnight)
""")


def cumulative_return(df_in, asset_col, start_idx, h):
    """Sum of asset returns from start_idx+1 to start_idx+h (inclusive)."""
    end_idx = min(start_idx + h, len(df_in) - 1)
    window  = df_in.loc[start_idx + 1 : end_idx, asset_col]
    if len(window) < h or window.isna().any():
        return np.nan
    return window.sum()


def run_backtest(df_in, pm_col, asset_name, asset_col, h_hold=1, tc=0.0001):
    shock_rows     = df_in.index[df_in[f"{pm_col}_shock"] == 1].tolist()
    trade_returns  = []
    next_available = 0

    for idx in shock_rows:
        if idx < next_available:
            continue                          # inside previous holding window
        signal = np.sign(df_in.loc[idx, pm_col])
        if signal == 0:
            continue
        cum_ret = cumulative_return(df_in, asset_col, idx, h_hold)
        if np.isnan(cum_ret):
            continue                          # NaN window (e.g. SPY overnight)
        trade_returns.append(signal * cum_ret)
        next_available = idx + h_hold

    if len(trade_returns) < 10:
        return None

    active   = np.array(trade_returns)
    n        = len(active)
    mean_ret = active.mean()
    std_ret  = active.std()

    gross_sr  = np.sqrt(ANNUAL_HOURS) * mean_ret / (std_ret + 1e-10)
    net_sr    = np.sqrt(ANNUAL_HOURS) * (mean_ret - tc) / (std_ret + 1e-10)
    cum_curve = np.cumsum(active)
    max_dd    = (cum_curve - np.maximum.accumulate(cum_curve)).min()
    dir_acc   = (active > 0).mean()
    binom_p   = stats.binomtest(int((active > 0).sum()), n, 0.5,
                                alternative="greater").pvalue

    return {"pm": pm_col, "asset": asset_name, "h": h_hold,
            "n_trades": n, "mean_ret": mean_ret, "cumulative_ret": active.sum(),
            "gross_sharpe": gross_sr, "net_sharpe": net_sr,
            "max_drawdown": max_dd, "dir_accuracy": dir_acc,
            "binom_p": binom_p, "_returns": active}

print(f"  {'PM Signal':25s}  {'Asset':10s}  {'h':>2}  {'Trades':>6}  "
      f"{'Gross SR':>9}  {'Net SR':>8}  {'MaxDD':>8}  {'DirAcc':>7}  {'p(dir)':>8}")
print(f"  {'-'*108}")

backtest_results = []
asset_order = list(CORE_ASSETS.keys()) + [k for k in EXTENDED if k in ASSETS]

for asset_name in asset_order:
    for pm in PM_FEATURES:
        key = (pm, asset_name)
        if key not in HOLDING_PERIODS:
            continue
        h         = HOLDING_PERIODS[key]
        asset_col = ASSETS[asset_name]
        tc        = TC.get(asset_name, 0.0002)
        result    = run_backtest(df, pm, asset_name, asset_col, h_hold=h, tc=tc)
        if result is None:
            continue
        backtest_results.append(result)
        r    = result
        flag = "✓" if r["net_sharpe"] > 0.5 else ("~" if r["net_sharpe"] > 0 else "✗")
        dstar = ("***" if r["binom_p"]<0.01 else "**" if r["binom_p"]<0.05
                 else "*" if r["binom_p"]<0.10 else "")
        tag  = " [ext]" if asset_name in EXTENDED else ""
        print(f"  {r['pm']:25s}  {asset_name+tag:10s}  {r['h']:2d}  "
              f"{r['n_trades']:6d}  "
              f"{r['gross_sharpe']:+9.3f}  {r['net_sharpe']:+8.3f} {flag}  "
              f"{r['max_drawdown']:+8.4f}  {r['dir_accuracy']:7.1%}  "
              f"{r['binom_p']:8.4f} {dstar}")


# 500-simulation random benchmark

print("\n500-simulation random direction benchmark:")
print(f"  {'PM → Asset':42s}  {'Mean Rand SR':>13}  {'Our Gross SR':>13}  {'Edge':>8}")
print(f"  {'-'*82}")

np.random.seed(42)
N_SIM = 500

for r in backtest_results:
    abs_rets = np.abs(r["_returns"])
    n        = len(abs_rets)
    sim_srs  = []
    for _ in range(N_SIM):
        rd     = np.random.choice([-1, 1], size=n)
        sr     = np.sqrt(ANNUAL_HOURS) * (rd * abs_rets).mean() / ((rd * abs_rets).std() + 1e-10)
        sim_srs.append(sr)
    mean_rand = np.mean(sim_srs)
    edge      = r["gross_sharpe"] - mean_rand
    label     = f"{r['pm']} → {r['asset']}"
    print(f"  {label:42s}  {mean_rand:+9.3f} ±{np.std(sim_srs):.2f}  "
          f"{r['gross_sharpe']:+13.3f}  {edge:+8.3f}")

for r in backtest_results:
    r.pop("_returns", None)

if backtest_results:
    pd.DataFrame(backtest_results).to_csv("backtest_results.csv", index=False)
    print("\nSaved: backtest_results.csv")




Assets: ['BTC', 'Gold', 'Oil', 'SPY', 'QQQ', 'SP500_fut']
Dataset: 9673 rows, 2025-03-05 → 2026-04-12

Run ML_analysis.py first to get optimal IRF holding periods.
  FED_DELTA                      → BTC           h=2 [default]
  FED_DELTA                      → Gold          h=2 [default]
  FED_DELTA                      → QQQ           h=2 [default]
  FED_DELTA                      → SP500_fut     h=2 [default]
  FED_DELTA                      → SPY           h=2 [default]
  INF_MONTHLY_DELTA              → Oil           h=1 [default]
  UNEMPLOYMENT_DELTA             → QQQ           h=1 [default]
  UNEMPLOYMENT_DELTA             → SP500_fut     h=1 [default]
  UNEMPLOYMENT_DELTA             → SPY           h=1 [default]

  SHOCK CONSTRUCTION (formula — aligned with Section 3B)
  FED_DELTA                     : 826 shocks (8.5%)
  GDP_DELTA                     : 838 shocks (8.7%)
  UNEMPLOYMENT_DELTA            : 740 shocks (7.7%)
  INF_MONTHLY_DELTA             : 724 shocks (7.5%)

  